# Main Dome: accumulated position errors

The purpose of this notebook is to plot the residual of the commanded and actual position of the dome in azimuth to look for any offsets between these quantities.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.time import Time

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

In [ ]:
def query_position_data(start, end):
    df_position = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.azimuth",
        columns=[f"positionActual"] + [f"positionCommanded"],
        begin=start,
        end=end,
    )

    return df_position

In [ ]:
def query_command_timestamps(start, end):
    df_command_az = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_moveAz",
        columns=[f"position"],
        begin=start,
        end=end,
    )

    timestamps = []
    for i in range(1,len(df_command_az)):
        if df_command_az["position"].iloc[i]-df_command_az["position"].iloc[i-1]>0.1:
            timestamps.append(pd.Timestamp(df_command_az["position"].index[i]))

    return timestamps

In [ ]:
def plot_position_residuals(df,start,end,t_start_period):
    """
    Plot commanded and current azimuth of the dome as a function of time,
    and corresponding residuals with respect to the commanded position.

    Args:
    df: dataframe with positional information
    start: positional index of df for start of time period
    end: positional index of df for end of time period
    t_start_period: timestamp used for title

    """
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))

    ax1.plot(df["positionActual"].iloc[start:end], label="Actual position")
    ax1.plot(df["positionCommanded"].iloc[start:end], label="Commanded position")
    ax1.set_ylabel("Azimuth (degrees)")
    ax1.legend()

    ax2.plot(
        df["positionActual"].iloc[start:end] - df["positionCommanded"].iloc[start:end],
        marker="o",
        markersize=1,
       linestyle="None",
    )
    ax2.set_ylim(-0.1, 0.2)
    ax2.set_ylabel("Residual (degrees)")
    ax2.set_xlabel("UTC")

    fig.autofmt_xdate()
    plt.suptitle(
        f"Azimuth commanded/registered dome position {t_start_period.to_value('iso', subfmt='date')}"
    )

In [ ]:
def plot_position_residuals_overplot(df,timestamps,t_start_period):
    """
    Plot commanded and actual azimuth of the dome residuals using the
    same reference timestamp (overplotting different subperiods)

    Args:
    df: dataframe with positional information
    timestamps: list with time stamps at which commands were given, to define subperiods
    t_start_period: timestamp used for title

    """
    fig, ax = plt.subplots()
    for i in range(len(timestamps)-1):
        start = df.index.searchsorted(timestamps[i], side='right')
        end = df.index.searchsorted(timestamps[i+1], side='right')
        residuals = df["positionActual"].iloc[start:end] - df["positionCommanded"].iloc[start:end]
        arbitrary_timestamps = np.arange(0,len(residuals))
        ax.plot(arbitrary_timestamps, residuals,
            marker="o",
            markersize=1,
            linestyle="None",
        )
    ax.set_ylim(-0.1, 0.2)
    ax.set_ylabel("Residual Current Az - Commanded Az (degrees)")
    ax.set_xlabel("Registered EFD events since position is commanded")
    plt.suptitle(
        f"Residual registered vs commanded dome azimuth {t_start_period.to_value('iso', subfmt='date_hm')}"
    )    

In [ ]:
# dome endurance test times
# t_start_period = Time("2025-03-16T10:39:00Z", scale="utc")
# t_end_period = Time("2025-03-27T11:39:00Z", scale="utc")
t_start_period = Time("2025-04-16T00:30:00Z", scale="utc")
t_end_period = Time("2025-04-16T04:00:00Z", scale="utc")

efd_client = makeEfdClient()

In [ ]:
# retrieve position data
df = query_position_data(t_start_period, t_end_period)

In [ ]:
# retrieve timestamp information when new moveAz commands are issued
command_timestamps = query_command_timestamps(t_start_period, t_end_period)

In [ ]:
plot_position_residuals(df, 0, len(df), t_start_period)

In [ ]:
plot_position_residuals_overplot(df, command_timestamps,t_start_period)